# Alzheimer's OASIS-1 — ANFIS 白盒诊断模块

输入 3 个特征：`[hippo_vol, wbv_final, mmse]`，先对 `hippo_vol`/`wbv_final` 做关于年龄的残差化
（缓解 `alzheimer_dsnet_oasis.ipynb` 里已经发现的年龄混淆：年龄匹配后 AUC 从 0.80 掉到 0.65），
再喂给一个自建的 PyTorch 原生 ANFIS（经典 Jang 五层结构：高斯隶属函数 → 乘积 T-norm →
归一化 → 一阶 TSK 线性后件 → 加权求和），回归到 CDR 数值刻度 {0, 0.5, 1, 2}，
推理时按阈值分桶为 Nondemented/Demented。

两个训练配置做消融（都用同一个 wbv_final，因为全脑体积已经没有第二个独立来源可比，
消融只聚焦在海马体积这一个变量上）：
- **ANFIS-on-GT**：`hippo_col=aseg_hippo_total`，用 FreeSurfer 真值海马体积训练，衡量 FIS 本身的性能上限
- **ANFIS-on-extracted-hippo**：`hippo_col=hippo_vol_hippodeep`，用 HippoDeep 的预测值训练，
  衡量海马体积特征提取误差对最终诊断的影响

5 折分层交叉验证，按 `label`（Nondemented/Demented）分层；每折训练完导出规则表和
汇总的 out-of-fold 预测，供下一个 notebook（`alzheimer_eval_anfis_colab.ipynb`）评估。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'pandas', 'scikit-learn', 'imbalanced-learn', 'torch'])

In [ ]:
import itertools
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

DRIVE_ROOT = '/content/drive/MyDrive'
OUTPUT_DIR = f'{DRIVE_ROOT}/alzheimer_anfis_outputs'
MASTER_CSV = f'{OUTPUT_DIR}/oasis_master_features_labels.csv'
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoints'
RESULTS_DIR = f'{OUTPUT_DIR}/results'
for d in [CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

N_FOLDS = 5
USE_SMOTE = True
N_EPOCHS = 300
LR = 5e-3
WEIGHT_DECAY = 1e-4
# 27 条规则、样本量有限时，落在数据稀疏格子里的规则后件系数几乎没有约束，
# 求值时容易外推出离谱的 severity（实测出现过样本量=2的格子给出 severity=4.63，
# 和其余规则完全不在一个量级）。额外对 consequent 参数做 L2 收缩，把这些
# 缺乏数据支持的系数往 0 拉，抑制这种外推。
CONSEQUENT_L2 = 1e-2

In [ ]:
master_df = pd.read_csv(MASTER_CSV)
print(f'主表 {len(master_df)} 例')
print(master_df['label'].value_counts())

# 两个消融配置：真值海马体积 vs HippoDeep 提取的海马体积。全脑体积两边都用 wbv_final
# （labels_merge notebook 里算好的：优先 aseg_wbv，缺失 fallback 到官方 nwbv_official）。
FEATURE_CONFIGS = {
    'ANFIS-on-GT': {
        'hippo_col': 'aseg_hippo_total',
        'wbv_col': 'wbv_final',
    },
    'ANFIS-on-extracted-hippo': {
        'hippo_col': 'hippo_vol_hippodeep',
        'wbv_col': 'wbv_final',
    },
}
for config_name, cfg in FEATURE_CONFIGS.items():
    missing = [c for c in (cfg['hippo_col'], cfg['wbv_col']) if c not in master_df.columns]
    if missing:
        print(f'⚠️ {config_name} 缺少列 {missing}——回 labels_merge notebook 检查 MASTER_CSV 是否正确生成。')
    else:
        n_valid = master_df[[cfg['hippo_col'], cfg['wbv_col']]].dropna().shape[0]
        print(f'{config_name}: 使用列 {cfg["hippo_col"]} / {cfg["wbv_col"]}，{n_valid} 例非空')

In [ ]:
# ── ANFIS 模块：经典 Jang 五层结构，PyTorch 原生实现 ────────────────

class GaussianMF(nn.Module):
    """Layer 1: fuzzification. 每个输入 n_mfs 个高斯隶属函数，中心/宽度可学习。"""

    def __init__(self, n_inputs: int, n_mfs: int, init_centers: np.ndarray, init_sigmas: np.ndarray):
        super().__init__()
        self.centers = nn.Parameter(torch.tensor(init_centers, dtype=torch.float32))  # [n_inputs, n_mfs]
        self.log_sigmas = nn.Parameter(torch.log(torch.tensor(init_sigmas, dtype=torch.float32) + 1e-6))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n_inputs] -> mu: [batch, n_inputs, n_mfs]
        sigmas = self.log_sigmas.exp()
        x = x.unsqueeze(-1)
        mu = torch.exp(-0.5 * ((x - self.centers) / sigmas) ** 2)
        return mu


class ANFIS(nn.Module):
    """5 层 ANFIS：fuzzification -> 规则激活强度(乘积T-norm) -> 归一化 -> 一阶TSK后件 -> 加权求和"""

    def __init__(self, n_inputs: int, n_mfs: int, init_centers: np.ndarray, init_sigmas: np.ndarray):
        super().__init__()
        self.n_inputs = n_inputs
        self.n_mfs = n_mfs
        self.n_rules = n_mfs ** n_inputs
        self.mf = GaussianMF(n_inputs, n_mfs, init_centers, init_sigmas)

        rule_indices = list(itertools.product(range(n_mfs), repeat=n_inputs))  # 27 tuples for n_inputs=3,n_mfs=3
        self.register_buffer('rule_idx', torch.tensor(rule_indices, dtype=torch.long))  # [n_rules, n_inputs]
        self.rule_indices = rule_indices

        # 一阶 TSK 后件: 每条规则 (n_inputs 个线性系数 + 1 个偏置)
        self.consequent = nn.Parameter(torch.zeros(self.n_rules, n_inputs + 1))

    def firing_strengths(self, x: torch.Tensor) -> torch.Tensor:
        mu = self.mf(x)  # [batch, n_inputs, n_mfs]
        batch_size = x.shape[0]
        idx = self.rule_idx.T.unsqueeze(0).expand(batch_size, -1, -1)  # [batch, n_inputs, n_rules]
        mu_gathered = torch.gather(mu, 2, idx)  # [batch, n_inputs, n_rules]
        firing = mu_gathered.prod(dim=1)  # [batch, n_rules]
        return firing

    def forward(self, x: torch.Tensor):
        firing = self.firing_strengths(x)
        norm_firing = firing / (firing.sum(dim=1, keepdim=True) + 1e-9)  # [batch, n_rules]
        ones = torch.ones(x.shape[0], 1, device=x.device, dtype=x.dtype)
        x_aug = torch.cat([x, ones], dim=1)  # [batch, n_inputs+1]
        rule_outputs = x_aug @ self.consequent.T  # [batch, n_rules]
        y = (norm_firing * rule_outputs).sum(dim=1)  # [batch]
        return y, norm_firing


def init_mf_params(train_features: np.ndarray, n_mfs: int = 3):
    """用训练折的分位数初始化隶属函数中心/宽度。train_features: [N, n_inputs]"""
    n_inputs = train_features.shape[1]
    quantiles = np.linspace(20, 80, n_mfs)  # 20/50/80 分位数 (n_mfs=3 时)
    centers = np.zeros((n_inputs, n_mfs))
    sigmas = np.zeros((n_inputs, n_mfs))
    for i in range(n_inputs):
        col = train_features[:, i]
        centers[i] = np.percentile(col, quantiles)
        spread = np.percentile(col, 66) - np.percentile(col, 33)
        sigmas[i] = max(spread * 0.5, 1e-3)
    return centers, sigmas


print('ANFIS 模块定义完成。')

In [ ]:
# ── 术语命名（供规则表打印用）：体积类特征 Low/Medium/High，MMSE Poor/Fair/Good ──
# 注意：hippo/wbv 送进 ANFIS 前已经做过关于年龄的残差化，所以这里的 Low/Medium/High
# 指"相对同龄人偏低/偏高"，不是原始绝对体积的高低——规则文本里显式标注 (age-adj)，
# 避免被当成绝对体积读错。
TERM_NAMES = {
    'hippo': ['Low', 'Medium', 'High'],
    'wbv': ['Low', 'Medium', 'High'],
    'mmse': ['Poor', 'Fair', 'Good'],
}
DISPLAY_NAMES = {
    'hippo': 'hippo(age-adj)',
    'wbv': 'wbv(age-adj)',
    'mmse': 'mmse',
}
FEATURE_ORDER = ['hippo', 'wbv', 'mmse']

# 27 条规则里，样本量极少的格子（比如只有 2~3 个训练样本落在那个组合附近）
# 对应的 TSK 线性后件基本是在没什么数据约束的情况下拟合出来的，在 MF 中心点求值时
# 容易外推出离谱数字（实测出现过 severity=4.63 这种和其余规则明显不在一个量级的值，
# 对应的训练支持度只有 2 个样本）。这里用"训练集里对这条规则的归一化激活强度总和"
# 作为软支持度指标，供后面做低置信度标注、以及供 IEEE 论文里如实报告规则可信度。
MIN_SUPPORT_PCT = 1.5  # 27 条规则均匀分布时每条约 3.7%，明显低于这个值的视为证据不足


def compute_rule_support(model: 'ANFIS', X_train: np.ndarray) -> np.ndarray:
    """对训练集算每条规则的软支持度（归一化激活强度之和），返回长度 n_rules 的数组。"""
    with torch.no_grad():
        x_t = torch.tensor(X_train, dtype=torch.float32, device=next(model.parameters()).device)
        _, norm_firing = model(x_t)
    return norm_firing.sum(dim=0).cpu().numpy()


def export_rule_table(model: ANFIS, X_train: np.ndarray, feature_names=FEATURE_ORDER):
    """打印全部规则: IF hippo is {term} AND wbv is {term} AND mmse is {term} THEN severity~=f_i -> label
    附带每条规则的训练集软支持度，支持度过低的规则标注为证据不足，不建议直接当结论用。"""
    centers = model.mf.centers.detach().cpu().numpy()  # [n_inputs, n_mfs]
    consequent = model.consequent.detach().cpu().numpy()  # [n_rules, n_inputs+1]
    support = compute_rule_support(model, X_train)
    support_pct = 100.0 * support / support.sum()
    rows = []
    for r, idx_tuple in enumerate(model.rule_indices):
        center_point = np.array([centers[i, idx_tuple[i]] for i in range(len(feature_names))] + [1.0])
        severity = float(consequent[r] @ center_point)
        label = 'Demented' if severity > 0.25 else 'Nondemented'
        terms = [TERM_NAMES[feature_names[i]][idx_tuple[i]] for i in range(len(feature_names))]
        rule_str = ' AND '.join(f'{DISPLAY_NAMES[feature_names[i]]} is {terms[i]}' for i in range(len(feature_names)))
        confidence = '低(样本不足)' if support_pct[r] < MIN_SUPPORT_PCT else '正常'
        rows.append({
            'rule_id': r,
            'rule': f'IF {rule_str} THEN severity~={severity:.2f} -> {label}  [支持度{support_pct[r]:.1f}%, 置信度{confidence}]',
            'severity_at_center': severity, 'label': label,
            'support_pct': support_pct[r], 'confidence': confidence,
        })
    return pd.DataFrame(rows)


def explain_prediction(model: ANFIS, x_single: torch.Tensor, X_train: np.ndarray,
                        feature_names=FEATURE_ORDER, top_k: int = 3):
    """对单个样本，按激活强度排序展示 Top-K 触发规则，作为诊断解释文本。
    低支持度的规则会在解释文本里标注出来，避免过度解读。"""
    with torch.no_grad():
        y, norm_firing = model(x_single.unsqueeze(0))
    firing = norm_firing[0].cpu().numpy()
    support = compute_rule_support(model, X_train)
    support_pct = 100.0 * support / support.sum()
    top_idx = np.argsort(firing)[::-1][:top_k]
    centers = model.mf.centers.detach().cpu().numpy()
    consequent = model.consequent.detach().cpu().numpy()
    lines = [f'预测严重度分数: {y.item():.3f}  ->  {"Demented" if y.item() > 0.25 else "Nondemented"}']
    for rank, r in enumerate(top_idx, 1):
        idx_tuple = model.rule_indices[r]
        terms = [TERM_NAMES[feature_names[i]][idx_tuple[i]] for i in range(len(feature_names))]
        rule_str = ' AND '.join(f'{DISPLAY_NAMES[feature_names[i]]} is {terms[i]}' for i in range(len(feature_names)))
        center_point = np.array([centers[i, idx_tuple[i]] for i in range(len(feature_names))] + [1.0])
        f_i = float(consequent[r] @ center_point)
        conf_note = '（该规则训练样本较少，解释仅供参考）' if support_pct[r] < MIN_SUPPORT_PCT else ''
        lines.append(f'  Rule #{r} 激活强度 {firing[r]*100:.1f}%: IF {rule_str} THEN severity={f_i:.2f}{conf_note}')
    return '\n'.join(lines)


print('规则表导出 / 单样本解释函数定义完成。')

In [ ]:
# ── 训练一个配置（GT 或 extracted-hippo）的完整 5 折交叉验证 ────

def train_one_config(config_name: str, hippo_col: str, wbv_col: str):
    print(f'\n{"="*60}\n[{config_name}]\n{"="*60}')

    df = master_df.dropna(subset=[hippo_col, wbv_col, 'mmse', 'age', 'cdr_numeric']).reset_index(drop=True)
    print(f'该配置可用样本数: {len(df)}（缺特征/缺MMSE的受试者已剔除）')

    X_raw = df[[hippo_col, wbv_col, 'mmse']].to_numpy(dtype=np.float32)
    age = df['age'].to_numpy(dtype=np.float32)
    cdr_numeric = df['cdr_numeric'].to_numpy(dtype=np.float32)
    label_binary = (df['label'] == 'Demented').astype(int).to_numpy()

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(df), dtype=np.float32)
    oof_fold = np.full(len(df), -1, dtype=int)
    fold_models = []
    rule_tables = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_raw, label_binary)):
        # 1) 年龄残差化：只在训练折上拟合线性回归，避免信息泄漏
        age_reg_hippo = LinearRegression().fit(age[train_idx].reshape(-1, 1), X_raw[train_idx, 0])
        age_reg_wbv = LinearRegression().fit(age[train_idx].reshape(-1, 1), X_raw[train_idx, 1])

        X_resid = X_raw.copy()
        X_resid[:, 0] = X_raw[:, 0] - age_reg_hippo.predict(age.reshape(-1, 1))
        X_resid[:, 1] = X_raw[:, 1] - age_reg_wbv.predict(age.reshape(-1, 1))
        # mmse (第3列) 不做年龄残差化，保留原始值

        # 1.5) 标准化：hippo 残差(~千mm^3量级)、wbv 残差(~0.05量级)、mmse(~14-30量级)
        # 三个特征原始尺度相差好几个数量级。不标准化的话有两个连带问题：
        # (a) SMOTE 找最近邻是按欧氏距离算的，尺度最大的 hippo 会完全主导距离计算，
        #     wbv/mmse 基本被忽略；(b) TSK 后件是对原始尺度的线性回归，不同特征的
        #     回归系数天然相差几个数量级，统一的 L2 正则起不到均衡收缩的作用。
        # 用训练折的 mean/std 标准化，同样的变换应用到验证折，避免信息泄漏。
        scaler = StandardScaler().fit(X_resid[train_idx])
        X_std = scaler.transform(X_resid)

        X_train, X_test = X_std[train_idx], X_std[test_idx]
        y_train, y_test = cdr_numeric[train_idx], cdr_numeric[test_idx]
        label_train = label_binary[train_idx]

        # 2) 类别不平衡处理：3D 标准化特征空间上对 Demented 做 SMOTE 过采样
        if USE_SMOTE:
            minority_count = min(np.bincount(label_train))
            k_neighbors = max(1, min(5, minority_count - 1))
            if minority_count > k_neighbors:
                sm = SMOTE(k_neighbors=k_neighbors, random_state=SEED)
                X_train_res, label_train_res = sm.fit_resample(X_train, label_train)
                # 合成样本的 cdr_numeric 目标：用原始训练样本做 1-NN 回归插值
                knn = KNeighborsRegressor(n_neighbors=1).fit(X_train, y_train)
                y_train_res = knn.predict(X_train_res)
            else:
                X_train_res, y_train_res, label_train_res = X_train, y_train, label_train
        else:
            X_train_res, y_train_res, label_train_res = X_train, y_train, label_train

        # 3) 类别加权（无论是否 SMOTE，都按逆频率再加一层权重，双重保险应对残余不平衡）
        class_counts = np.bincount(label_train_res, minlength=2)
        class_weight = len(label_train_res) / (2.0 * np.maximum(class_counts, 1))
        sample_weight = class_weight[label_train_res]

        # 4) 隶属函数初始化 + 训练
        centers, sigmas = init_mf_params(X_train_res, n_mfs=3)
        model = ANFIS(n_inputs=3, n_mfs=3, init_centers=centers, init_sigmas=sigmas).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.SmoothL1Loss(reduction='none')

        X_train_t = torch.tensor(X_train_res, dtype=torch.float32, device=DEVICE)
        y_train_t = torch.tensor(y_train_res, dtype=torch.float32, device=DEVICE)
        w_train_t = torch.tensor(sample_weight, dtype=torch.float32, device=DEVICE)
        X_test_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)

        model.train()
        for epoch in range(N_EPOCHS):
            optimizer.zero_grad()
            pred, _ = model(X_train_t)
            data_loss = (criterion(pred, y_train_t) * w_train_t).mean()
            consequent_l2 = model.consequent.pow(2).sum()
            loss = data_loss + CONSEQUENT_L2 * consequent_l2
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 100 == 0:
                print(f'  [{config_name}][fold {fold}] epoch {epoch+1}/{N_EPOCHS} '
                      f'loss={loss.item():.4f} (data={data_loss.item():.4f})')

        model.eval()
        with torch.no_grad():
            test_pred, _ = model(X_test_t)
        oof_pred[test_idx] = test_pred.cpu().numpy()
        oof_fold[test_idx] = fold

        # 存 checkpoint 时把 scaler 的 mean_/scale_ 也存进去——推理一个新受试者时
        # 需要用同样的标准化参数处理原始特征，否则直接喂给 model 会因为尺度不对而预测错误。
        ckpt_path = f'{CHECKPOINT_DIR}/anfis_{config_name.replace(" ", "_")}_fold{fold}.pth'
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler_mean': scaler.mean_,
            'scaler_scale': scaler.scale_,
        }, ckpt_path)
        fold_models.append(model)
        rule_tables.append(export_rule_table(model, X_train_res))
        print(f'  [{config_name}][fold {fold}] 完成，测试折 {len(test_idx)} 例，checkpoint: {ckpt_path}')

    result_df = df.copy()
    result_df['oof_pred_severity'] = oof_pred
    result_df['oof_fold'] = oof_fold
    result_df['oof_pred_label'] = np.where(oof_pred > 0.25, 'Demented', 'Nondemented')
    result_df['config'] = config_name

    oof_csv = f'{RESULTS_DIR}/oof_predictions_{config_name.replace(" ", "_")}.csv'
    result_df.to_csv(oof_csv, index=False)
    print(f'保存 out-of-fold 预测: {oof_csv}')

    # 保存最后一折的规则表作为代表性示例（各折规则表都存了，这里额外单独存一份方便直接看）
    rule_tables[-1].to_csv(f'{RESULTS_DIR}/rule_table_{config_name.replace(" ", "_")}_fold{N_FOLDS-1}.csv', index=False)

    return result_df, fold_models, rule_tables

In [ ]:
results = {}
models = {}
rules = {}
for config_name, cfg in FEATURE_CONFIGS.items():
    missing = [c for c in (cfg['hippo_col'], cfg['wbv_col']) if c not in master_df.columns]
    if missing:
        print(f'跳过 {config_name}：主表里缺少列 {missing}（回 labels_merge notebook 检查 MASTER_CSV）')
        continue
    result_df, fold_models, rule_tables = train_one_config(config_name, cfg['hippo_col'], cfg['wbv_col'])
    results[config_name] = result_df
    models[config_name] = fold_models
    rules[config_name] = rule_tables

print('\n全部配置训练完成。out-of-fold 预测已存到', RESULTS_DIR)

In [ ]:
# ── 打印其中一个配置的完整规则表，人工核查 Low/Medium/High 划分是否符合医学直觉 ──
for config_name, rule_tables in rules.items():
    print(f'\n=== {config_name}: fold {N_FOLDS-1} 的 27 条规则 ===')
    for _, row in rule_tables[-1].iterrows():
        print(row['rule'])